# Day 1 · Exercise 3: Inspect a Video with ffprobe

**What you'll build:** `inspect_video(path)` — a function that extracts metadata from any video file using `ffprobe` and returns it as a clean Python dict.

**Why it matters:** Every time you generate a video programmatically — from SadTalker, FFmpeg, or any other tool — you need to verify the output is what you expected. `inspect_video` is the function you'll reach for throughout the course. It's also your first practice at calling external tools from Python using `subprocess`, a pattern that appears constantly in AI engineering pipelines.

**How to complete this exercise:**
1. Read the docstring — the required keys and types are specified there
2. Replace `pass` with your implementation
3. Run the **Check Your Work** cell — all 5 checks must pass

> **Hint if you're stuck:** `ffprobe -v quiet -print_format json -show_streams <path>` — parse the JSON, find the stream where `codec_type == "video"`, and extract the values.

## Your Implementation

In [ ]:
import subprocess
import json

def inspect_video(path: str) -> dict:
    """
    Extract metadata from a video file using ffprobe.

    Args:
        path: Path to the video file.

    Returns:
        A dict with exactly these keys:
            duration  (float)  — total length in seconds
            width     (int)    — horizontal resolution in pixels
            height    (int)    — vertical resolution in pixels
            fps       (float)  — frames per second, rounded to 3 decimal places
            codec     (str)    — video codec name, e.g. 'h264'

    Implementation notes:
        - Run: ffprobe -v quiet -print_format json -show_streams <path>
        - Parse result.stdout as JSON
        - Find the stream where codec_type == "video"
        - fps is stored as a fraction string like "30/1" or "30000/1001" in
          the r_frame_rate field — split on "/" and divide
        - duration is a string in the JSON — cast to float
        - width and height are already ints in the JSON

    Example:
        info = inspect_video("lesson.mp4")
        print(info)
        # {'duration': 72.4, 'width': 1280, 'height': 720,
        #  'fps': 25.0, 'codec': 'h264'}
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    pass
    # ────────────────────────────────────────────────────────────────────────

## Check Your Work

Run the cell below. It generates a short test video, runs your function on it, and checks all five returned values.

In [ ]:
import os, subprocess

_PASS = '\u2705'
_FAIL = '\u274c'
_TEST_VIDEO = '__check_inspect.mp4'

def _make_test_video():
    """Generate a 3-second 320x240 30fps test video using FFmpeg."""
    result = subprocess.run(
        [
            "ffmpeg", "-y",
            "-f", "lavfi",
            "-i", "testsrc=duration=3:size=320x240:rate=30",
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            _TEST_VIDEO,
        ],
        capture_output=True,
    )
    return result.returncode == 0 and os.path.exists(_TEST_VIDEO)

def _run_checks():
    score = 0
    total = 5

    # Setup — generate test video
    if not _make_test_video():
        print(f'{_FAIL} Could not generate test video — is FFmpeg installed?')
        return

    # Check 1: function is callable
    try:
        assert callable(inspect_video), 'inspect_video is not defined or not callable'
        print(f'{_PASS} Check 1/5: function exists and is callable')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 1/5: {e}')
        return

    # Check 2: returns a dict with all required keys
    try:
        result = inspect_video(_TEST_VIDEO)
        assert isinstance(result, dict), f'Expected dict, got {type(result).__name__}'
        required = {'duration', 'width', 'height', 'fps', 'codec'}
        missing = required - result.keys()
        assert not missing, f'Missing keys: {missing}'
        print(f'{_PASS} Check 2/5: returns a dict with all 5 required keys')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/5: {e}')
        return

    # Check 3: correct resolution
    try:
        assert result['width']  == 320, f"width: expected 320, got {result['width']}"
        assert result['height'] == 240, f"height: expected 240, got {result['height']}"
        print(f"{_PASS} Check 3/5: correct resolution ({result['width']}x{result['height']})")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/5: {e}')

    # Check 4: correct fps (within 0.5 of 30)
    try:
        fps = result['fps']
        assert isinstance(fps, float), f'fps should be float, got {type(fps).__name__}'
        assert abs(fps - 30.0) < 0.5, f'fps: expected ~30.0, got {fps}'
        print(f"{_PASS} Check 4/5: correct fps ({fps})")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/5: {e}')

    # Check 5: correct duration (within 0.5s of 3.0)
    try:
        dur = result['duration']
        assert isinstance(dur, float), f'duration should be float, got {type(dur).__name__}'
        assert abs(dur - 3.0) < 0.5, f'duration: expected ~3.0s, got {dur}'
        print(f"{_PASS} Check 5/5: correct duration ({dur:.2f}s)")
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 5/5: {e}')

    # Result
    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 3 complete! All {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} checks passed. Keep going — you\'re close!')

    if os.path.exists(_TEST_VIDEO):
        os.remove(_TEST_VIDEO)

_run_checks()

## Bonus Challenge

Once all 5 checks pass, use `inspect_video` on a real output from the pipeline:

```python
info = inspect_video("../../00_pipeline/day_001_final.mp4")
for k, v in info.items():
    print(f"{k:10s}: {v}")
```

What's the actual duration of the Day 1 final video? Is the fps what you'd expect? What codec did FFmpeg choose?

This is the kind of sanity check you'd add to the end of any video generation pipeline — verify the output before calling it done.

In [ ]:
# Bonus: inspect the pipeline output
# info = inspect_video("../../00_pipeline/day_001_final.mp4")
# for k, v in info.items():
#     print(f"{k:10s}: {v}")

---
## Solution

<details>
<summary>Click to reveal solution — try on your own first</summary>

```python
def inspect_video(path: str) -> dict:
    result = subprocess.run(
        [
            "ffprobe", "-v", "quiet",
            "-print_format", "json",
            "-show_streams",
            path,
        ],
        capture_output=True,
        text=True,
    )
    data = json.loads(result.stdout)
    video = next(
        s for s in data["streams"] if s["codec_type"] == "video"
    )
    num, den = map(int, video["r_frame_rate"].split("/"))
    return {
        "duration": float(video["duration"]),
        "width":    int(video["width"]),
        "height":   int(video["height"]),
        "fps":      round(num / den, 3),
        "codec":    video["codec_name"],
    }
```

**Why this works:**
- `ffprobe -print_format json -show_streams` outputs all stream metadata as JSON. A typical video file has two streams: video and audio.
- We use `next(s for s in ... if s["codec_type"] == "video")` to find the video stream specifically.
- `r_frame_rate` is a rational number stored as a string like `"30/1"` or `"30000/1001"` (the latter is NTSC's ~29.97 fps). We split and divide to get a float.
- `duration` in the stream metadata is already in seconds, but it's a string — we cast to float.

**Why subprocess over a Python library:**  
`ffprobe` is the authoritative source for media metadata. Python libraries like `moviepy` or `opencv` extract metadata by decoding part of the file, which is slower and occasionally wrong. `ffprobe` reads the container headers directly — it's fast, reliable, and works on every format FFmpeg supports.

</details>